# N164 · 线程、互斥与顺序同步

**目标：** 用同步关系保证顺序而不是依靠启动次序。

**先修：** N009, N053, N013。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** threading；共享状态；race；Lock/Event；happens-before；GIL非同步契约。

**配套讲解来源：** [同名逐章意见](../../comment/164_threads_ordering_events.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

LeetCode 1114「Print in Order」的场景是：有一个类暴露 `first`、`second`、`third` 三个方法，三个线程会各自调用其中一个，但**调用（启动）顺序完全随机**。你要保证三个回调的实际执行顺序永远是"先 first、再 second、后 third"，输出恒为 `firstsecondthird`。

举个例子：操作系统可能先调度了执行 `third` 的线程，接着是 `second` 的线程，最后才是 `first` 的线程。如果什么都不做，输出可能是 `thirdsecondfirst`，彻底乱套。我们要的是一种**同步关系**：不管谁先被启动，`second` 必须等 `first` 的回调真正跑完才能开始，`third` 必须等 `second` 跑完才能开始。

一个直观但错误的做法是"睡一会儿"——比如 `third` 里 `time.sleep(0.01)` 赌 `first` 已经跑完。这在慢机器、高负载下随时翻车。正确做法是用线程间通信原语表达"等这件事做完"，而不是"等这么多毫秒"。“思路推导”部分 里那句话就是这个意思：这是一条执行先后关系，不是"睡一会儿大概够了"。

## 2. 基础知识：先读懂这些词

- **线程（threading.Thread）**：程序里一条独立执行的指令流。同一个进程里的多个线程共享内存，所以它们能看见同一份变量，也正因为如此才会互相踩踏。
- **共享状态**：多个线程都能读写的变量。本章的 `out` 列表被三个线程同时 append，就是共享状态。
- **竞态（race condition）**：结果取决于线程被调度执行的先后次序。无同步版本输出乱序，就是竞态的直接表现。
- **锁（threading.Lock）**：最基础的互斥工具，同一时刻只允许一个线程持有。本章的 `run_threads` 用 `error_lock` 保护 `errors` 列表，避免多个线程同时 append 出问题。
- **事件（threading.Event）**：一个带内部布尔标志的线程间信号。`set()` 把标志置为真并唤醒所有等待者；`wait()` 在标志为假时阻塞、为真时立即通过。关键性质：**标志有记忆**（latch，门闩语义）——先 `set()` 后 `wait()` 也不会丢失通知，`wait()` 直接放行。这和"一次性的响铃"不同。
- **happens-before（先行发生）**：如果操作 A 完成是操作 B 开始的前提，那 A 一定先于 B 发生。我们用两个 Event 构造出 `first回调 → second回调 → third回调` 的链条。
- **GIL（全局解释器锁）非同步契约**：CPython 的 GIL 保证同一时刻只有一个线程执行字节码，但这不保证线程之间的先后顺序，也不能替代锁和事件——"不会数据撕裂"不等于"顺序正确"。
- **join 与线程泄漏**：`thread.join()` 等一个线程彻底结束。测试最后 `assert threading.active_count()==1`（只剩主线程），就是在检查没有线程泄漏地挂着。
- **隔离进程测试**：把可能卡死的整组线程放进子进程里跑，外层设 20 秒超时；超时就杀掉子进程并让测试失败——超时不算成功，也不用 daemon 线程掩盖卡死。

## 3. 思路推导：从小例子开始

**Step 1：把"顺序"翻译成两个"事件"。** 我们需要两个带记忆的信号：`first_done`（first 的回调完成了）和 `second_done`（second 的回调完成了）。三个方法的逻辑就是：

- `first`：直接执行回调，然后 `first_done.set()`。
- `second`：先 `first_done.wait()`，再执行回调，然后 `second_done.set()`。
- `third`：先 `second_done.wait()`，再执行回调。

**Step 2：手推一个"最坏启动顺序"。** 假设线程启动顺序是 `[third, second, first]`（“运行示例”部分 表格里 6 种排列之一）：

```
时刻    third 线程             second 线程           first 线程
T1     second_done.wait() → 阻塞（标志为假）
T2                            first_done.wait() → 阻塞
T3                                                  printFirst() 执行，输出 'first'
T4                                                  first_done.set() → 标志置真
T5                            被唤醒，wait() 返回
T6                            printSecond() 输出 'second'
T7                            second_done.set()
T8     被唤醒，wait() 返回
T9     printThird() 输出 'third'
最终输出：firstsecondthird
```

**Step 3：事件先触发后等待也不怕。** 如果调度反过来、`first` 线程最先跑完并 `set()` 了，之后 `second` 才走到 `wait()`——因为标志已经是真，`wait()` 立即通过，通知不会丢。这就是选 Event 而不是"发一次性信号"的原因。

**Step 4：正确性传递。** `second` 的回调只在 `first_done` 为真之后运行，而 `first_done` 为真只在 `first` 回调之后；同理 `third` 在 `second` 之后。两级等待串起来，按传递性得到完整顺序——这和 “正确性与复杂度”部分 的论证一致。

**Step 5：测试基建。** `run_threads` 为每个函数开一个线程，用 `guarded` 包一层捕获异常（异常经 `error_lock` 保护后收集，任一线程炸了最后统一 raise）；全部 `join` 之后断言没有线程还活着。`run_ordering_case(start_order)` 校验启动排列合法（0、1、2 各出现一次），按给定顺序把三个方法丢进 `run_threads`，返回拼接后的输出。“运行示例”部分 的表格把 6 种排列全跑了一遍，输出清一色 `firstsecondthird`；“自动测试”部分 再把这套东西循环 10 轮防"运气好"。

## 4. 核心代码：Foo

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
class Foo:

    def __init__(self):
        self.first_done = threading.Event()
        self.second_done = threading.Event()

    def first(self, printFirst):
        printFirst()
        self.first_done.set()

    def second(self, printSecond):
        self.first_done.wait()
        printSecond()
        self.second_done.set()

    def third(self, printThird):
        self.second_done.wait()
        printThird()
```

### 逐段读懂代码

“分段实现”部分 是一段 Python 源码（`THREAD_SOURCE` 字符串），由 `run_isolated` 放进子进程执行；notebook 内核里也 `exec` 了一份供直接调用。

**run_threads（测试基建）：**

（对应程序见下方分段实现与完整代码。）

`guarded` 把每个函数包起来抓 `BaseException`（连 `KeyboardInterrupt` 这类都抓），抓到后用 `error_lock` 互斥地放进 `errors`——不加锁的话"三个线程同时 append"本身就是一个竞态。先全部 `start()` 再逐个 `join()`，保证三个线程确实并发启动。如果有任何异常，把第一个重新抛出让测试失败；最后那句 assert 是"所有线程都 join 干净"的兜底检查。docstring 提醒调用者：可能阻塞的用例要放到隔离进程里跑。

**Foo（本章核心，就 15 行）：**

（对应程序见下方分段实现与完整代码。）

`__init__` 建两个 Event，初始都是假。`first` 不等任何人：执行回调后 `set()`，向 `second` 发出"我完成了"。`second` 的三行是"等→做→发信号"：先 `wait()` 阻塞到 `first_done` 为真，执行回调，再 `set()` 放行 `third`。`third` 只等不发。注意每个方法都假定"每个方法只被调用一次"，这正是题面给的运行条件。

**run_ordering_case：**

```python
def run_ordering_case(start_order):
    if sorted(start_order) != [0, 1, 2]:
        raise ValueError('Each method must be launched exactly once')
    foo = Foo()
    out = []
    methods = [lambda: foo.first(lambda: out.append('first')), lambda: foo.second(lambda: out.append('second')), lambda: foo.third(lambda: out.append('third'))]
    run_threads([methods[i] for i in start_order])
    return ''.join(out)
```

第一行是边界防御：启动排列必须是 0、1、2 的某个排列，否则直接 ValueError。`out` 是共享输出缓冲，三个回调分别往里 append 各自的名字。`methods[i]` 把"方法下标"映射到对应的调用函数，按 `start_order` 重排后交给 `run_threads`。最后把列表拼成字符串返回，方便断言比较。

## 5. 分段实现：按顺序运行

**本章接口：** `Foo.first/second/third`、`run_ordering_case(start_order)`

### THREAD_SOURCE

In [1]:
THREAD_SOURCE = r'''import threading
from collections import deque

def run_threads(functions):
    """Caller must run potentially blocking cases inside the isolated process."""
    errors = []
    error_lock = threading.Lock()
    def guarded(fn):
        try:
            fn()
        except BaseException as error:
            with error_lock:
                errors.append(error)
    threads = [threading.Thread(target=guarded, args=(fn,)) for fn in functions]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    if errors:
        raise errors[0]
    assert not any(thread.is_alive() for thread in threads)

class Foo:
    def __init__(self):
        self.first_done = threading.Event()
        self.second_done = threading.Event()
    def first(self, printFirst):
        printFirst()
        self.first_done.set()
    def second(self, printSecond):
        self.first_done.wait()
        printSecond()
        self.second_done.set()
    def third(self, printThird):
        self.second_done.wait()
        printThird()

def run_ordering_case(start_order):
    if sorted(start_order) != [0,1,2]:
        raise ValueError('Each method must be launched exactly once')
    foo = Foo()
    out = []
    methods = [lambda: foo.first(lambda: out.append('first')),
               lambda: foo.second(lambda: out.append('second')),
               lambda: foo.third(lambda: out.append('third'))]
    run_threads([methods[i] for i in start_order])
    return ''.join(out)
'''

### 导入本章使用的库

In [2]:
import json
import subprocess
import sys

### 初始化与接口绑定

In [3]:
exec(THREAD_SOURCE, globals())

### run_isolated

In [4]:
def run_isolated(extra):
    """A timeout is a failure; subprocess.run kills and reaps the child."""
    result = subprocess.run([sys.executable, '-c', THREAD_SOURCE + '\n' + extra], text=True, capture_output=True, check=True, timeout=20)
    return result.stdout

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [5]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

from IPython.display import Code
display(Code(THREAD_SOURCE,language='python'))
rows = json.loads(run_isolated("import json,itertools; print(json.dumps([[list(p),run_ordering_case(p)] for p in itertools.permutations(range(3))]))"))
show_table(['thread start order','actual callback order'],rows)

import threading
from collections import deque

def run_threads(functions):
    """Caller must run potentially blocking cases inside the isolated process."""
    errors = []
    error_lock = threading.Lock()
    def guarded(fn):
        try:
            fn()
        except BaseException as error:
            with error_lock:
                errors.append(error)
    threads = [threading.Thread(target=guarded, args=(fn,)) for fn in functions]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    if errors:
        raise errors[0]
    assert not any(thread.is_alive() for thread in threads)

class Foo:
    def __init__(self):
        self.first_done = threading.Event()
        self.second_done = threading.Event()
    def first(self, printFirst):
        printFirst()
        self.first_done.set()
    def second(self, printSecond):
        self.first_done.wait()
        printSecond()
        self.second_done.set()
    def third(self, printThird):
        self.second_done.wait()
        printThird()

def run_ordering_case(start_order):
    if sorted(start_order) != [0,1,2]:
        raise ValueError('Each method must be launched exactly once')
    foo = Foo()
    out = []
    methods = [lambda: foo.first(lambda: out.append('first')),
               lambda: foo.second(lambda: out.append('second')),
               lambda: foo.third(lambda: out.append('third'))]
    run_threads([methods[i] for i in start_order])
    return ''.join(out)

thread start order,actual callback order
"[0, 1, 2]",firstsecondthird
"[0, 2, 1]",firstsecondthird
"[1, 0, 2]",firstsecondthird
"[1, 2, 0]",firstsecondthird
"[2, 0, 1]",firstsecondthird
"[2, 1, 0]",firstsecondthird


## 7. 正确性、适用条件与复杂度

A 只能在 first 回调之后触发，second 只能在 A 之后运行；同理 third 在 second 之后。因此由传递性保证完整顺序，各方法启动一次保证回调各一次。活性另要求所有方法都被调用、回调有限完成、线程最终获得调度。

**代价：** 同步状态 O(1)。等待时间由任务和调度决定，不应声称固定毫秒延迟；调度测试有限次通过不等于证明所有环境都及时响应。

### 展开理解

**正确性：** 关键事实有三个。第一，`first_done` 只在 `first` 的回调执行完之后才可能为真（`set()` 写在回调后面）；第二，`second` 的回调只在 `first_done` 为真之后才运行（`wait()` 写在回调前面）；两者合起来就是"second 回调必然晚于 first 回调"。third 与 second 之间是同样的关系，串起来按传递性得到完整顺序。第三，因为每个方法只被调用一次，三个回调各执行一次，输出长度也对。活性（最终都能跑完）还需要附加前提：三个方法都被调用、回调有限完成、线程最终获得调度——如果有人永远不调 `first`，`second` 就会永远等下去，这不是算法 bug 而是输入没满足。

**复杂度：** 同步状态只有两个 Event，O(1) 空间。等待时长由任务本身和操作系统调度决定，任何"固定多少毫秒"的说法都不成立。另外 “正确性与复杂度”部分 特意提醒：调度相关的测试跑有限次通过，不等于证明了所有环境下都及时响应——所以测试循环了 10 轮只是增加信心，不是证明。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的测试代码在子进程里执行：

```python
for _ in range(10):
    for order in itertools.permutations(range(3)):
        assert run_ordering_case(order) == 'firstsecondthird'

assert threading.active_count() == 1
```

- `itertools.permutations(range(3))` 枚举 6 种启动排列（“运行示例”部分 的表格就是它的输出，每行是"启动顺序 → 实际输出"）：这是穷举边界，说明无论谁先启动结果都一样。
- 外层 `for _ in range(10)`：重复 10 轮，覆盖不同的线程调度运气——单次通过可能只是碰巧。
- 断言值 `'firstsecondthird'`：回调顺序恰好是先 first 后 second 再 third，一个字符都不能错位。
- `threading.active_count()==1`：所有工作线程都被 join 回收，没有泄漏线程（若 second/third 卡死在 wait，join 就不会返回，子进程会超时被杀，测试失败而不是"侥幸通过"）。
- 从测试基建看，`run_threads` 里任何一个线程抛异常都会导致 `raise errors[0]`，加上子进程 20 秒超时兜底——"卡死"和"异常"都算失败，绝不算成功。

In [6]:
print(run_isolated("import itertools\nfor _ in range(10):\n    for order in itertools.permutations(range(3)):\n        assert run_ordering_case(order) == 'firstsecondthird'\nassert threading.active_count() == 1\nprint('All six start orders passed; every worker joined.')\n"))

print('N164: 所有本章断言通过')

All six start orders passed; every worker joined.

N164: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 枚举三线程启动排列。

**练习 2：** 构造无同步版本的可能交错，而不要求随机必现错误。

<details>
<summary>展开思路提示（不是完整答案）</summary>

**练习 1（枚举三线程启动排列）：** 提示：排列就是 `itertools.permutations` 给出的 6 种。对每种排列画一条和第三节那样的线程时间线，标出谁在 `wait()` 里阻塞、哪个 `set()` 唤醒了谁。手算完可以用 `run_ordering_case` 逐个验证，预测和结果应当完全一致。边界别忘了"排列不合法要报错"（比如 `[0,0,1]`）。

**练习 2（构造无同步版本的交错）：** 提示：把 `Foo` 里的两个 Event 拿掉、回调直接执行，就是无同步版。你要做的是**构造**一种可能的交错并论证它输出错误，而不是赌跑一万次必然出错——比如给出"third 线程先被调度完整执行"的时刻表，推出输出 `third...`。可以借助"主线程先 start 哪个、操作系统先跑哪个"的组合来描述，不必真的让错误随机复现。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

本章包含真实的 SQL／Shell／JavaScript／线程程序，Python 部分负责创建示例、调用运行时和断言。嵌入的程序源码也在下面完整显示；缺少相应运行时应安装依赖，不能用模拟结果替代。

In [7]:
# 完整实现：本单元格不依赖其他单元格。
THREAD_SOURCE = r'''import threading
from collections import deque

def run_threads(functions):
    """Caller must run potentially blocking cases inside the isolated process."""
    errors = []
    error_lock = threading.Lock()
    def guarded(fn):
        try:
            fn()
        except BaseException as error:
            with error_lock:
                errors.append(error)
    threads = [threading.Thread(target=guarded, args=(fn,)) for fn in functions]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    if errors:
        raise errors[0]
    assert not any(thread.is_alive() for thread in threads)

class Foo:
    def __init__(self):
        self.first_done = threading.Event()
        self.second_done = threading.Event()
    def first(self, printFirst):
        printFirst()
        self.first_done.set()
    def second(self, printSecond):
        self.first_done.wait()
        printSecond()
        self.second_done.set()
    def third(self, printThird):
        self.second_done.wait()
        printThird()

def run_ordering_case(start_order):
    if sorted(start_order) != [0,1,2]:
        raise ValueError('Each method must be launched exactly once')
    foo = Foo()
    out = []
    methods = [lambda: foo.first(lambda: out.append('first')),
               lambda: foo.second(lambda: out.append('second')),
               lambda: foo.third(lambda: out.append('third'))]
    run_threads([methods[i] for i in start_order])
    return ''.join(out)
'''

import json

import subprocess

import sys

exec(THREAD_SOURCE, globals())

def run_isolated(extra):
    """A timeout is a failure; subprocess.run kills and reaps the child."""
    result = subprocess.run([sys.executable, '-c', THREAD_SOURCE + '\n' + extra], text=True, capture_output=True, check=True, timeout=20)
    return result.stdout

# 示例与回归测试
print(run_isolated("import itertools\nfor _ in range(10):\n    for order in itertools.permutations(range(3)):\n        assert run_ordering_case(order) == 'firstsecondthird'\nassert threading.active_count() == 1\nprint('All six start orders passed; every worker joined.')\n"))

print('N164: 所有本章断言通过')

All six start orders passed; every worker joined.

N164: 所有本章断言通过


## 11. 代表题与迁移

- **1114. Print in Order（canonical）**：就是本章的原题——三个方法、三个线程、任意启动顺序，用两个 Event（或两个 Semaphore、一个 Condition）构造 happens-before 链，保证输出恒为 `firstsecondthird`。

**题面入口：** [1114](https://leetcode.com/problems/print-in-order/)

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。